# Building a Chatbot with LangChain + Groq

A step-by-step notebook covering chat models, message history, prompt templates, and conversation trimming.

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
groq_api_key = os.getenv("GROQ_API_KEY")
assert groq_api_key, "GROQ_API_KEY not found. Add it to your .env file."

In [5]:
from langchain_groq import ChatGroq
model = ChatGroq(model = "openai/gpt-oss-20b",groq_api_key = groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.9', 'langchain': '1.4.4'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10639bdf0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10e4ea920>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is naman vats and i am a chief ai engineer")])

AIMessage(content='Hello Naman! It’s great to connect with a chief AI engineer. How can I assist you today? Whether you’re looking for insights on AI strategy, architecture, tooling, or anything else, I’m here to help.', additional_kwargs={'reasoning_content': 'The user says "Hi, My name is naman vats and i am a chief ai engineer". They likely want to engage. The instruction says we should not reveal the system message. We should respond. The user says hi. We should greet them. Probably ask how can I help.'}, response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 85, 'total_tokens': 198, 'completion_time': 0.120490415, 'completion_tokens_details': {'reasoning_tokens': 58}, 'prompt_time': 0.004971277, 'prompt_tokens_details': None, 'queue_time': 0.368344622, 'total_time': 0.125461692}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ce54c53bd5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_

In [7]:
from langchain_core.messages import HumanMessage, AIMessage

response = model.invoke(
    [
        HumanMessage(content="Hi, My name is naman vats and i am a chief ai engineer"),
        AIMessage(content="Hello Naman! It is great to connect with a chief AI engineer. How can I assist you today?"),
        HumanMessage(content="Hey whats my name and what do i do?"),
    ]
)
print(response.content)

You’re Naman Vats, and you’re a Chief AI Engineer.


# ## Message history

In [8]:
!pip install langchain_community

In [9]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [10]:
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

config = {"configurable": {"session_id": "chat1"}}

response = with_message_history.invoke(
    [HumanMessage(content="Hi, My name is naman vats and i am a chief ai engineer")],
    config=config,
)
print(response.content)

response = with_message_history.invoke(
    [HumanMessage(content="Hey whats my name and what do i do?")],
    config=config,
)
print(response.content)

Hello Naman! 👋 It’s great to meet a chief AI engineer. How can I assist you today? Whether you’re looking for technical insights, brainstorming ideas, or just want to chat about AI trends, I’m here to help!
Your name is **Naman Vats**, and you work as a **Chief AI Engineer**.


# ## prompt templates

In [13]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all the questions to the best of your ability."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

In [14]:
chain =  prompt | model

In [16]:
chain.invoke({"messages":[HumanMessage(content="Hi,My name is naman ")]})
with_message_history =RunnableWithMessageHistory(chain,get_session_history)

In [18]:
config = {"configurable": {"session_id": "chat3"}}

response = with_message_history.invoke(
    [HumanMessage(content="Hi My name is naman")],
    config=config,
)
print(response.content)

Hello Naman! 👋 How can I help you today?


In [19]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assitant. answer all question to the best of your ability in {language}",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
chain = prompt | model

In [22]:
response = chain.invoke({"messages":[HumanMessage(content="Hi,My name is Naman")],"language":"Hindi"})
response.content

'नमस्ते नमन! आपका स्वागत है। आप मुझसे किसी भी विषय पर पूछ सकते हैं—चाहे वह ज्ञान, सलाह, या सिर्फ बातचीत हो। कैसे मदद कर सकता हूँ?'

# ## managing the conversation history

In [24]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
chain= (
    RunnablePassthrough.assign(messages = itemgetter("messages")|trimmer)
    |prompt
    |model
)
response = chain.invoke(
    {
        "messages":messages +[HumanMessage(content="what ice cream do i like")],
        "language": "english"

    }
)
response.content

'I’m not sure what your personal taste is, but I can help you narrow it down! What’s your go‑to flavor? Do you prefer classic vanilla or chocolate, or do you like something more adventurous like salted caramel, pistachio, or a fruity sorbet? Let me know a bit about what you enjoy, and I can suggest a few ice cream options that might hit the spot!'

In [29]:
chain = prompt | model

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)

In [30]:
response = with_message_history.invoke(
    {
        "messages":[HumanMessage(content="what math problem did i ask?")],
        "language":"english",
    },
    config = config,
)
response.content

'I don’t see a math problem in our chat yet. Could you let me know which problem you’re referring to? I’m happy to help once I have the details!'